# Citation-support scoring: Colab rerun launcher

**New, unexecuted launcher created 18 November 2026.** The original executed notebook was not supplied. Genuine prior Colab execution evidence is documented in docs/colab_reproduction_report.md. This notebook acquires the pinned archive, executes unchanged source, compares results and exports logs. CPU is sufficient. Read data/README.md before acquiring data; public release terms remain unresolved.

Upload the GitHub repository ZIP or evaluation ZIP when prompted. Do not upload the raw archive to a public notebook. Download outputs before the runtime ends.


In [ ]:
from google.colab import files
uploaded = files.upload()


In [ ]:
from pathlib import Path
import zipfile, os, sys, subprocess, hashlib, json, platform
from datetime import datetime, timezone
zips = [Path(name).resolve() for name in uploaded if name.lower().endswith('.zip')]
assert len(zips) == 1, 'Upload one project ZIP.'
root = Path('/content/atr_rerun_' + datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S_%f'))
with zipfile.ZipFile(zips[0]) as z:
    assert z.testzip() is None
    for name in z.namelist():
        assert (root / name).resolve().is_relative_to(root.resolve())
    z.extractall(root)
entrypoints = list(root.rglob('scripts/run_all.py'))
assert len(entrypoints) == 1
PROJECT = entrypoints[0].parents[1]
os.chdir(PROJECT)
assert sys.version_info >= (3, 11)
print(PROJECT, sys.version)


In [ ]:
DATASET = Path('/content/atr_private_data/human_evaluation_annotations.jsonl')
if not DATASET.exists():
    subprocess.run([sys.executable, 'scripts/acquire_data.py', '--destination', str(DATASET)], check=True)
assert DATASET.stat().st_size == 17520509
assert hashlib.sha256(DATASET.read_bytes()).hexdigest() == '51c66d4dcec9b6bc035d9a08e51d4a7b750615b6fc469be1d666e5d5693ebb05'
print('Input identity verified')


In [ ]:
RUN_OUT = PROJECT / 'results' / ('rerun_' + datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S_%f'))
cmd = [sys.executable, 'scripts/run_all.py', '--data', str(DATASET), '--out', str(RUN_OUT)]
r = subprocess.run(cmd, capture_output=True, text=True)
print(r.stdout + r.stderr)
if (RUN_OUT / 'logs').exists():
    (RUN_OUT / 'logs/launcher_console.txt').write_text(r.stdout + r.stderr)
    context = {'execution_location': 'Google Colab', 'python': sys.version, 'platform': platform.platform(), 'command': cmd, 'exit_code': r.returncode, 'note': 'Overrides the interpretive meaning of the inherited hardcoded environment label; original manifest is preserved.'}
    (RUN_OUT / 'logs/launcher_context.json').write_text(json.dumps(context, indent=2))
assert r.returncode == 0, 'Stop and inspect the actual execution logs.'


In [ ]:
import csv
canonical = PROJECT / 'results/clean_run'
checks = {}
for actual in sorted(RUN_OUT.rglob('*.csv')):
    relative = actual.relative_to(RUN_OUT)
    # Public copy intentionally contains only aggregate canonical tables.
    expected = canonical / relative
    if expected.exists():
        checks[str(relative)] = actual.read_bytes() == expected.read_bytes()
assert checks and all(checks.values()), checks
print(json.dumps(checks, indent=2))
(RUN_OUT / 'logs/canonical_comparison.json').write_text(json.dumps(checks, indent=2))
for row in csv.DictReader((RUN_OUT / 'tables/system_scores.csv').open()):
    print(row['system'], row['C0_percent'], row['C1_percent'], row['loss_pp'])


In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt'], check=True)
r = subprocess.run([sys.executable, 'scripts/generate_figures.py', '--out', str(RUN_OUT)], capture_output=True, text=True)
(RUN_OUT / 'logs/plotting_console.txt').write_text(r.stdout + r.stderr)
assert r.returncode == 0, r.stderr
from IPython.display import display, Image
for name in ['precision', 'gap_changes', 'loss_ecdf', 'robustness']:
    display(Image(filename=str(RUN_OUT / 'figures' / (name + '.png'))))


In [ ]:
import shutil
from importlib.metadata import distributions
packages = sorted([{'name': p.metadata.get('Name', 'UNKNOWN'), 'version': p.version} for p in distributions()], key=lambda p: p['name'].lower())
(RUN_OUT / 'logs/package_versions.json').write_text(json.dumps(packages, indent=2))
evidence = shutil.make_archive('/content/ATR_Evidence_' + RUN_OUT.name, 'zip', root_dir=RUN_OUT.parent, base_dir=RUN_OUT.name)
files.download(evidence)
print('Also save File > Download > .ipynb, with actual outputs.')
